# Lab 5, going further: from the tiny GPT to the models you call

**Optional. About 15 minutes to run on a GPU. Read the outputs; you do not need to follow the code.**

The model at the end of `gpt_dev.ipynb` has 209,729 parameters and reads one character at a time. The models you call through OpenRouter have billions of parameters. This notebook climbs from one to the other in four steps.

Every step uses the recipe you saw in the tour: guess the next token, measure the loss, nudge the numbers, repeat. What changes from step to step is the size of the model, the amount of text, and what counts as a token.

| Step | Model | Size | Reads | Where it comes from |
|---|---|---|---|---|
| 0 | `gpt_dev.ipynb`, last cell | 209,729 parameters | characters | trained in minutes on a laptop |
| 1 | the same design with the dials turned up | about 10.7 million | characters | **trained here**, a few minutes on a fast GPU |
| 2 | TinyStories-33M | printed below | word pieces | downloaded; too slow to train in class |
| 3 | OLMo 2 1B, from the Allen Institute for AI | printed below | word pieces | downloaded, at four points during its training |
| 4 | Llama 3.2 1B and a larger model | 1 billion and up | word pieces | called through OpenRouter with your course key |

**Before you start:** choose a GPU with *Runtime → Change runtime type*. An A100 (Colab Pro) is fastest. A T4 also works, more slowly.

*Drafted by Claude (Anthropic), based on the direction of Xiuye Chen.*

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none. Choose a GPU runtime before continuing.")

## Step 1. Turn the dials up, and train it here

The tour's slide "The dials a person set" has a button for the video's bigger run: each character described by 384 numbers, 6 blocks, and a window of 256 characters. That is about 10.7 million parameters, 50 times the notebook's model. It is trained on the same megabyte of Shakespeare, split the same way.

We use nanoGPT, Andrej Karpathy's faster version of the same code. Its README reports about 3 minutes for this run on one A100, ending at a loss of 1.47 on held-back text. The notebook's own model ended near 1.82. Lower is better.

In [ ]:
!git clone -q https://github.com/karpathy/nanoGPT.git
%cd nanoGPT
!pip -q install tiktoken
!python data/shakespeare_char/prepare.py

In [ ]:
# The training run. Watch the "val loss" lines: that is the loss on held-back text.
!python train.py config/train_shakespeare_char.py

# No GPU? This smaller version from the nanoGPT README runs on a CPU in about 3 minutes and reaches a loss near 1.88:
# !python train.py config/train_shakespeare_char.py --device=cpu --compile=False --eval_iters=20 --log_interval=1 --block_size=64 --batch_size=12 --n_layer=4 --n_head=4 --n_embd=128 --max_iters=2000 --lr_decay_iters=2000 --dropout=0.0

In [ ]:
# Let the trained model write. (Add --device=cpu if you used the CPU version above.)
!python sample.py --out_dir=out-shakespeare-char --num_samples=2 --max_new_tokens=400
%cd ..

**Look at:** real words, speaker names, verse-like lines. It still does not mean anything. More parameters and a longer window bought better spelling and form, from the same small text.

## Step 2. More text, and word pieces in place of characters

TinyStories is a collection of roughly two million very short stories written with a small child's vocabulary. A model trained on it reads **word pieces**, the kind of token real models use, and it has far more text to learn from than one megabyte.

Training a model like this takes hours even on several A100s, so we download one that is already trained. The code below is the writing loop from the tour: look at the text so far, get odds for the next token, draw one, add it, repeat.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("roneneldan/TinyStories-33M")
tiny = AutoModelForCausalLM.from_pretrained("roneneldan/TinyStories-33M")
print("parameters:", f"{sum(p.numel() for p in tiny.parameters()):,}")
print("vocabulary:", f"{tok.vocab_size:,}", "word pieces")

prompt = "Once upon a time, a little robot wanted to learn how to read."
torch.manual_seed(1710)
ids = tok(prompt, return_tensors="pt")
out = tiny.generate(**ids, max_new_tokens=120, do_sample=True, temperature=0.8, top_k=50, pad_token_id=tok.eos_token_id)
print()
print(tok.decode(out[0], skip_special_tokens=True))

**Look at:** whole sentences that follow from one another. Change the prompt and run the cell again. Then try a prompt that is not a children's story, such as `The capital of France is`. The model only knows the world of its training text.

(The "33M" in the name does not count the table of word pieces, which is why the number printed above is larger.)

## Step 3. A fully open model, watched while it learns

OLMo is a family of language models from the Allen Institute for AI. Its training code, its training data and hundreds of checkpoints saved during training are all public. That lets us do for a real model what the tour's "Watch it learn" slide did for the tiny one.

The cell below loads the smallest OLMo 2 model at four points in its training and gives each the same prompt. Each checkpoint is a 6 GB download, so this cell takes a few minutes.

In [ ]:
import gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

NAME = "allenai/OLMo-2-0425-1B"
CHECKPOINTS = [
    ("stage1-step0-tokens0B",        "before any training"),
    ("stage1-step300-tokens1B",      "after about 1 billion tokens"),
    ("stage1-step10000-tokens21B",   "after about 21 billion tokens"),
    ("main",                         "finished: about 4 trillion tokens"),
]
prompt = "The capital of France is"

device = "cuda" if torch.cuda.is_available() else "cpu"
olmo_tok = AutoTokenizer.from_pretrained(NAME)
for revision, when in CHECKPOINTS:
    model = AutoModelForCausalLM.from_pretrained(NAME, revision=revision, torch_dtype=torch.float16 if device == "cuda" else torch.float32).to(device)
    if revision == CHECKPOINTS[0][0]:
        print("parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
    torch.manual_seed(1710)
    ids = olmo_tok(prompt, return_tensors="pt", return_token_type_ids=False).to(device)
    out = model.generate(**ids, max_new_tokens=40, do_sample=True, temperature=0.8, top_k=50)
    print()
    print(f"--- {when} ---")
    print(olmo_tok.decode(out[0], skip_special_tokens=True))
    del model; gc.collect(); torch.cuda.empty_cache()

**Look at:** the same arc as the tiny model, at a very different scale. Noise first, then common words, then grammar, then facts. The first billion tokens is already a thousand times more text than the Shakespeare file.

## Step 4. The models you call

The last step is not something we can download and run in a lab. It is the kind of model you called in Lab 4. Send the same prompt to a 1-billion-parameter model and to a much larger one, and compare.

This uses your course key, stored as the Colab secret `OPENROUTER_API_KEY`, exactly as in the Lab 4 notebook. The two calls together cost a fraction of a cent.

In [ ]:
%pip -q install "openai>=1.0"
import os
from openai import OpenAI

try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise ValueError("Add OPENROUTER_API_KEY as a Colab secret, then run again.")

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)
question = "Finish this sentence in one line, then say in one more line how sure you are: The capital of France is"

for model_id in ["meta-llama/llama-3.2-1b-instruct", "openai/gpt-6-luna"]:
    response = client.chat.completions.create(model=model_id, messages=[{"role": "user", "content": question}], max_tokens=120)
    print(f"--- {model_id} ---")
    print(response.choices[0].message.content)
    print("tokens in:", response.usage.prompt_tokens, "| tokens out:", response.usage.completion_tokens)
    print()

## What changed on the way up, and what did not

| | Step 0 | Step 1 | Step 2 | Step 3 | Step 4 |
|---|---|---|---|---|---|
| Parameters | 0.2 million | 10.7 million | tens of millions | over a billion | a billion and up |
| Token | character | character | word piece | word piece | word piece |
| Training text | 1 MB | 1 MB | about 2 million short stories | about 4 trillion tokens | not always published |
| Can we train it in class? | yes, minutes | yes, minutes on an A100 | no, hours on several GPUs | no | no |

**The same at every step:** text becomes tokens, the model is trained to guess the next token, attention moves information between tokens, and writing is the loop of guess, draw, append.

**One more step that this notebook leaves out:** the models in Step 4 were also trained on conversations, which is what makes them answer a question where the others just continue the text.

*A note on OLMo and OpenRouter: OpenRouter lists larger OLMo models, but on October 7, 2026 no provider was serving them, so Step 4 uses Llama 3.2 1B.*